<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/column_value_count_chatGpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from functools import reduce

spark = SparkSession.builder.getOrCreate()

# Input Data
data = [
    (10, 20, 11, 20),
    (20, 11, 10, 99),
    (10, 11, 20, 1),
    (30, 12, 20, 99),
    (10, 11, 20, 20),
    (40, 13, 15, 3),
    (30, 8, 11, 99)
]

df = spark.createDataFrame(data, ["A", "B", "C", "D"])

df.show()

+---+---+---+---+
|  A|  B|  C|  D|
+---+---+---+---+
| 10| 20| 11| 20|
| 20| 11| 10| 99|
| 10| 11| 20|  1|
| 30| 12| 20| 99|
| 10| 11| 20| 20|
| 40| 13| 15|  3|
| 30|  8| 11| 99|
+---+---+---+---+



In [ ]:
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window
from functools import reduce

window = Window.orderBy("value")

result_dfs = []

for c in df.columns:
    tmp = (
        df.groupBy(c)
          .count()
          .withColumnRenamed(c, "value")
          .withColumnRenamed("count", f"count_{c}")
          .withColumn("rn", row_number().over(window))
          .withColumnRenamed("value", c)
    )

    result_dfs.append(tmp)

final_df = reduce(lambda x, y: x.join(y, "rn", "full"), result_dfs).drop("rn")

final_df.show()

+----+-------+---+-------+----+-------+----+-------+
|   A|count_A|  B|count_B|   C|count_C|   D|count_D|
+----+-------+---+-------+----+-------+----+-------+
|  10|      3|  8|      1|  10|      1|   1|      1|
|  20|      1| 11|      3|  11|      2|   3|      1|
|  30|      2| 12|      1|  15|      1|  20|      2|
|  40|      1| 13|      1|  20|      3|  99|      3|
|NULL|   NULL| 20|      1|NULL|   NULL|NULL|   NULL|
+----+-------+---+-------+----+-------+----+-------+

